<a href="https://colab.research.google.com/github/divyadharshini-1306/Mutual-Fund-Style-Drifter/blob/main/Phase2_CapClassification_Join.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [1]:
from google.colab import drive
drive.mount('/content/drive')

import pandas as pd

holdings = pd.read_csv("/content/drive/MyDrive/StyleDrift_Project/processed/holdings_master.csv")
print(holdings.shape)
holdings.head()

Mounted at /content/drive
Mounted at /content/drive
(22461, 10)
(22461, 10)


,instrument_name,isin,sector,quantity,market_value_lakhs,pct_to_aum,scheme_name,statement_date,amc,file_label
0,HDFC Bank Ltd.,INE040A01034,Banks,60500000.0,450452.75,8.42,SBI Large Cap Fund,2026-05-31 00:00:00,SBI,Monthly Portfolio Disclosure - May 2026
1,ICICI Bank Ltd.,INE090A01021,Banks,32850000.0,412727.40,7.71,SBI Large Cap Fund,2026-05-31 00:00:00,SBI,Monthly Portfolio Disclosure - May 2026
2,Reliance Industries Ltd.,INE002A01018,Petroleum Products,24500000.0,323694.00,6.05,SBI Large Cap Fund,2026-05-31 00:00:00,SBI,Monthly Portfolio Disclosure - May 2026
3,Larsen & Toubro Ltd.,INE018A01030,Construction,7400000.0,301661.00,5.64,SBI Large Cap Fund,2026-05-31 00:00:00,SBI,Monthly Portfolio Disclosure - May 2026
4,Asian Paints Ltd.,INE021A01026,Consumer Durables,8300000.0,221742.80,4.14,SBI Large Cap Fund,2026-05-31 00:00:00,SBI,Monthly Portfolio Disclosure - May 2026


,instrument_name,isin,sector,quantity,market_value_lakhs,pct_to_aum,scheme_name,statement_date,amc,file_label
0,HDFC Bank Ltd.,INE040A01034,Banks,60500000.0,450452.75,8.42,SBI Large Cap Fund,2026-05-31 00:00:00,SBI,Monthly Portfolio Disclosure - May 2026
1,ICICI Bank Ltd.,INE090A01021,Banks,32850000.0,412727.40,7.71,SBI Large Cap Fund,2026-05-31 00:00:00,SBI,Monthly Portfolio Disclosure - May 2026
2,Reliance Industries Ltd.,INE002A01018,Petroleum Products,24500000.0,323694.00,6.05,SBI Large Cap Fund,2026-05-31 00:00:00,SBI,Monthly Portfolio Disclosure - May 2026
3,Larsen & Toubro Ltd.,INE018A01030,Construction,7400000.0,301661.00,5.64,SBI Large Cap Fund,2026-05-31 00:00:00,SBI,Monthly Portfolio Disclosure - May 2026
4,Asian Paints Ltd.,INE021A01026,Consumer Durables,8300000.0,221742.80,4.14,SBI Large Cap Fund,2026-05-31 00:00:00,SBI,Monthly Portfolio Disclosure - May 2026


In [2]:
import os

cap_list_dir = "/content/drive/MyDrive/StyleDrift_Project/raw_data/amfi_cap_list"
print(os.listdir(cap_list_dir))

['Average_Market_Capitalization_30_Jun2024_2a1ab4c1d8.xlsx', 'AverageMarketCapitalizationoflistedcompaniesduringthesixmonthsended31Dec2024.xlsx', 'AverageMarketCapitalization31Dec2025.xlsx', 'AverageMarketCapitalization30Jun2025.xlsx', 'AverageMarketCapitalizationoflistedcompaniesduringthesixmonthsended30Jun2023.xlsx', 'AverageMarketCapitalizationoflistedcompaniesduringthesixmonthsended31Dec2023.xlsx', 'AverageMarketCapitalizationoflistedcompaniesduringthesixmonthsended31Dec2022.xlsx']


In [3]:
sample_file = os.path.join(cap_list_dir, "AverageMarketCapitalization30Jun2025.xlsx")

df_cap_sample = pd.read_excel(sample_file, sheet_name=None, header=None)
print("Sheets:", list(df_cap_sample.keys()))

first_sheet = list(df_cap_sample.keys())[0]
df_cap_sample[first_sheet].head(15)

Sheets: ['Sheet1']


,0,1,2,3,4,5,6,7,8,9,10
0,Average Market Capitalization of listed compan...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
1,Sr. No.,Company name,ISIN,BSE Symbol,BSE 6 month Avg Total Market Cap in (Rs. Crs.),NSE Symbol,NSE 6 month Avg Total Market Cap (Rs. Crs.),MSEI Symbol,MSEI 6 month Avg Total Market Cap in (Rs Crs.),Average of All Exchanges (Rs. Cr.),Categorization as per SEBI Circular dated Oct ...
2,1,Reliance Industries Ltd,INE002A01018,RELIANCE,1778085.84447,RELIANCE,1778158.8713,NaN,NaN,1778122.357885,Large Cap
3,2,HDFC Bank Ltd.,INE040A01034,HDFCBANK,1386322.170815,HDFCBANK,1385733.3714,NaN,NaN,1386027.771108,Large Cap
4,3,Tata Consultancy Services Ltd.,INE467B01029,TCS,1324318.278843,TCS,1324366.8142,NaN,NaN,1324342.546521,Large Cap
5,4,Bharti Airtel Ltd.,INE397D01024,BHARTIARTL,1046084.608199,BHARTIARTL,1045871.3918,NaN,NaN,1045977.999999,Large Cap
6,5,ICICI Bank Ltd.,INE090A01021,ICICIBANK,946977.716109,ICICIBANK,942729.7426,NaN,NaN,944853.729354,Large Cap
7,6,Infosys Ltd,INE009A01021,INFY,693147.410417,INFY,693080.3332,NaN,NaN,693113.871808,Large Cap
8,7,State Bank Of India,INE062A01020,SBIN,687254.151717,SBIN,687304.2167,NaN,NaN,687279.184208,Large Cap
9,8,Hindustan Unilever Ltd.,INE030A01027,HINDUNILVR,546321.22953,HINDUNILVR,546302.7002,NaN,NaN,546311.964865,Large Cap


In [4]:
# Each AMFI cap list covers a 6-month window.
# We parse all 4 files and tag each with the date range it covers.
# Later we'll pick the RIGHT one for each holding's statement_date.

import pandas as pd
import os

cap_list_dir = "/content/drive/MyDrive/StyleDrift_Project/raw_data/amfi_cap_list"

# Map each filename to the 6-month window it applies to
# Key = exact filename, Value = (window_start, window_end) as strings
cap_file_windows = {
    "AverageMarketCapitalizationduringthesixmonthsended31Dec2024.xlsx": ("2024-07-01", "2024-12-31"),
    "AverageMarketCapitalization30Jun2025.xlsx":                       ("2025-01-01", "2025-06-30"),
    "AverageMarketCapitalization31Dec2025.xlsx":                       ("2025-07-01", "2025-12-31"),
}

# Print what files you actually have — confirm filenames match above
print("Files in cap_list_dir:")
for f in os.listdir(cap_list_dir):
    print(" ", f)

Files in cap_list_dir:
  Average_Market_Capitalization_30_Jun2024_2a1ab4c1d8.xlsx
  AverageMarketCapitalizationoflistedcompaniesduringthesixmonthsended31Dec2024.xlsx
  AverageMarketCapitalization31Dec2025.xlsx
  AverageMarketCapitalization30Jun2025.xlsx
  AverageMarketCapitalizationoflistedcompaniesduringthesixmonthsended30Jun2023.xlsx
  AverageMarketCapitalizationoflistedcompaniesduringthesixmonthsended31Dec2023.xlsx
  AverageMarketCapitalizationoflistedcompaniesduringthesixmonthsended31Dec2022.xlsx


In [6]:
cap_file_windows = {
    "AverageMarketCapitalizationoflistedcompaniesduringthesixmonthsended31Dec2022.xlsx": ("2022-07-01", "2022-12-31"),
    "AverageMarketCapitalizationoflistedcompaniesduringthesixmonthsended30Jun2023.xlsx":  ("2023-01-01", "2023-06-30"),
    "AverageMarketCapitalizationoflistedcompaniesduringthesixmonthsended31Dec2023.xlsx":  ("2023-07-01", "2023-12-31"),
    "Average_Market_Capitalization_30_Jun2024_2a1ab4c1d8.xlsx":                           ("2024-01-01", "2024-06-30"),
    "AverageMarketCapitalizationoflistedcompaniesduringthesixmonthsended31Dec2024.xlsx":  ("2024-07-01", "2024-12-31"),
    "AverageMarketCapitalization30Jun2025.xlsx":                                           ("2025-01-01", "2025-06-30"),
    "AverageMarketCapitalization31Dec2025.xlsx":                                           ("2025-07-01", "2025-12-31"),
}

all_cap_dfs = []

for filename, (win_start, win_end) in cap_file_windows.items():
    filepath = os.path.join(cap_list_dir, filename)

    # Read raw with no header
    raw = pd.read_excel(filepath, header=None, sheet_name=0)

    # Skip title row (row 0) and header row (row 1), data starts row 2
    df = raw.iloc[2:].copy()

    # Assign column names — structure is identical across all 7 files
    df.columns = [
        "sr_no", "company_name", "isin",
        "bse_symbol", "bse_mktcap",
        "nse_symbol", "nse_mktcap",
        "msei_symbol", "msei_mktcap",
        "avg_mktcap", "cap_category"
    ]

    # Keep only what we need
    df = df[["isin", "company_name", "cap_category"]].copy()

    # Drop blank rows at the bottom of some files
    df = df.dropna(subset=["isin"])

    # Clean whitespace from category text
    df["cap_category"] = df["cap_category"].str.strip()

    # Tag with window
    df["window_start"] = pd.to_datetime(win_start)
    df["window_end"]   = pd.to_datetime(win_end)

    all_cap_dfs.append(df)
    print(f"✓ {filename[:60]}...: {len(df)} stocks, window {win_start} → {win_end}")

# Stack all 7 windows into one lookup table
cap_lookup = pd.concat(all_cap_dfs, ignore_index=True)
print(f"\ncap_lookup shape: {cap_lookup.shape}")
print(f"Windows covered: {cap_lookup['window_start'].dt.year.min()} → {cap_lookup['window_end'].dt.year.max()}")
print(f"Unique categories: {cap_lookup['cap_category'].unique()}")
cap_lookup.head(3)



✓ AverageMarketCapitalizationoflistedcompaniesduringthesixmont...: 5072 stocks, window 2022-07-01 → 2022-12-31
✓ AverageMarketCapitalizationoflistedcompaniesduringthesixmont...: 5058 stocks, window 2023-01-01 → 2023-06-30
✓ AverageMarketCapitalizationoflistedcompaniesduringthesixmont...: 5012 stocks, window 2023-07-01 → 2023-12-31
✓ Average_Market_Capitalization_30_Jun2024_2a1ab4c1d8.xlsx...: 5049 stocks, window 2024-01-01 → 2024-06-30
✓ AverageMarketCapitalizationoflistedcompaniesduringthesixmont...: 5097 stocks, window 2024-07-01 → 2024-12-31
✓ AverageMarketCapitalization30Jun2025.xlsx...: 5158 stocks, window 2025-01-01 → 2025-06-30
✓ AverageMarketCapitalization31Dec2025.xlsx...: 5372 stocks, window 2025-07-01 → 2025-12-31

cap_lookup shape: (35818, 5)
Windows covered: 2022 → 2025
Unique categories: ['Large Cap' 'Mid Cap' 'Small Cap']


,isin,company_name,cap_category,window_start,window_end
0,INE002A01018,Reliance Industries Ltd,Large Cap,2022-07-01,2022-12-31
1,INE467B01029,Tata Consultancy Services Ltd.,Large Cap,2022-07-01,2022-12-31
2,INE040A01034,HDFC Bank Ltd.,Large Cap,2022-07-01,2022-12-31


In [8]:
holdings = pd.read_csv(
    "/content/drive/MyDrive/StyleDrift_Project/processed/holdings_master.csv"
)

# format="mixed" handles both "2026-05-31 00:00:00" and "31-May-2026" automatically
holdings["statement_date"] = pd.to_datetime(holdings["statement_date"], format="mixed", dayfirst=True).dt.normalize()

# Check date range per fund
print("Date range per fund:")
print(
    holdings.groupby("scheme_name")["statement_date"]
            .agg(["min", "max", "nunique"])
            .rename(columns={"min":"earliest", "max":"latest", "nunique":"months"})
            .to_string()
)

# Check rows outside cap list window
outside_window = holdings[holdings["statement_date"] < "2022-07-01"]
print(f"\nRows before Jul 2022 (no cap list): {len(outside_window)}")
print(f"Rows within window: {len(holdings) - len(outside_window)}")

Date range per fund:
                            earliest     latest  months
scheme_name                                            
Kotak Flexicap Fund       2024-11-30 2026-05-31      19
Kotak Large & Midcap Fund 2024-11-30 2026-05-31      19
Kotak Large Cap Fund      2024-11-30 2026-05-31      19
Kotak Midcap Fund         2024-11-30 2026-05-31      19
Kotak Multicap Fund       2024-11-30 2026-05-31      19
Kotak Small Cap Fund      2024-11-30 2026-05-31      19
SBI Flexicap Fund         2023-01-31 2026-05-31      40
SBI Large Cap Fund        2023-01-31 2026-05-31      40
SBI Large and Midcap Fund 2023-01-31 2026-05-31      40
SBI Midcap Fund           2023-01-31 2026-05-31      40
SBI MultiCap Fund         2023-01-31 2026-05-31      40
SBI Smallcap Fund         2023-01-31 2026-05-31      40

Rows before Jul 2022 (no cap list): 0
Rows within window: 22461


In [9]:
from tqdm import tqdm
tqdm.pandas()

def get_cap_category(isin, stmt_date, cap_lookup):
    match = cap_lookup[
        (cap_lookup["isin"] == isin) &
        (cap_lookup["window_start"] <= stmt_date) &
        (cap_lookup["window_end"]   >= stmt_date)
    ]
    if len(match) > 0:
        return match.iloc[0]["cap_category"]
    else:
        return "Unclassified"

print("Applying cap classification (takes ~2-3 mins)...")
holdings["cap_category"] = holdings.progress_apply(
    lambda row: get_cap_category(row["isin"], row["statement_date"], cap_lookup),
    axis=1
)

print("\nCap category distribution:")
print(holdings["cap_category"].value_counts())
print(f"\nUnclassified %: {round(holdings['cap_category'].eq('Unclassified').mean() * 100, 2)}%")

Applying cap classification (takes ~2-3 mins)...


100%|██████████| 22461/22461 [02:41<00:00, 138.88it/s]


Cap category distribution:
cap_category
Small Cap       6674
Mid Cap         5762
Large Cap       5464
Unclassified    4561
Name: count, dtype: int64

Unclassified %: 20.31%


In [10]:
# Pick one unclassified row and investigate why it didn't match
unclassified_rows = holdings[holdings["cap_category"] == "Unclassified"]

# Look at a sample of unclassified ISINs
sample_isins = unclassified_rows["isin"].head(10).tolist()
print("Sample unclassified ISINs from holdings:")
for i in sample_isins:
    print(f"  repr: {repr(i)}")

print("\nSample ISINs from cap_lookup:")
for i in cap_lookup["isin"].head(5).tolist():
    print(f"  repr: {repr(i)}")

# Check if any of the unclassified ISINs exist in cap_lookup at all
# (ignoring the date window — just raw ISIN presence)
sample = sample_isins[0]
print(f"\nSearching for '{sample}' in cap_lookup ignoring window...")
print(cap_lookup[cap_lookup["isin"] == sample])

# Also check with strip — maybe whitespace is the issue
print(f"\nSearching with stripped ISIN...")
print(cap_lookup[cap_lookup["isin"].str.strip() == sample.strip()])

Sample unclassified ISINs from holdings:
  repr: 'INE040A01034'
  repr: 'INE090A01021'
  repr: 'INE002A01018'
  repr: 'INE018A01030'
  repr: 'INE021A01026'
  repr: 'INE009A01021'
  repr: 'INE062A01020'
  repr: 'INE775A01035'
  repr: 'INE238A01034'
  repr: 'INE795G01014'

Sample ISINs from cap_lookup:
  repr: 'INE002A01018'
  repr: 'INE467B01029'
  repr: 'INE040A01034'
  repr: 'INE009A01021'
  repr: 'INE030A01027'

Searching for 'INE040A01034' in cap_lookup ignoring window...
               isin    company_name cap_category window_start window_end
2      INE040A01034  HDFC Bank Ltd.    Large Cap   2022-07-01 2022-12-31
5074   INE040A01034  HDFC Bank Ltd.    Large Cap   2023-01-01 2023-06-30
10132  INE040A01034  HDFC BANK LTD.    Large Cap   2023-07-01 2023-12-31
15144  INE040A01034  HDFC Bank Ltd.    Large Cap   2024-01-01 2024-06-30
20193  INE040A01034  HDFC Bank Ltd.    Large Cap   2024-07-01 2024-12-31
25289  INE040A01034  HDFC Bank Ltd.    Large Cap   2025-01-01 2025-06-30
30447  IN

In [11]:
# Check what statement_dates the unclassified rows actually have
print("Statement dates of unclassified rows:")
print(unclassified_rows["statement_date"].value_counts().sort_index())

# Pick HDFC Bank specifically and check which dates it appears as unclassified
hdfc_unclassified = unclassified_rows[unclassified_rows["isin"] == "INE040A01034"]
print(f"\nHDFC Bank unclassified rows: {len(hdfc_unclassified)}")
print(hdfc_unclassified[["statement_date", "scheme_name"]].to_string())

# Check the exact window boundaries in cap_lookup
print("\nWindow boundaries in cap_lookup:")
print(cap_lookup[["window_start","window_end"]].drop_duplicates().sort_values("window_start").to_string())

Statement dates of unclassified rows:
statement_date
2023-01-31      7
2023-02-28     14
2023-03-31      7
2023-04-30      7
2023-05-31      7
2023-06-30      6
2023-07-31      6
2023-08-31      6
2023-09-30     11
2023-10-31      7
2023-11-30      7
2023-12-31      7
2024-01-31     10
2024-02-29     10
2024-03-31      9
2024-04-30      8
2024-05-31      7
2024-06-30      8
2024-07-31     12
2024-08-31     11
2024-09-30      8
2024-10-31      8
2024-11-30     16
2024-12-31     21
2025-01-31     26
2025-02-28     10
2025-03-31     29
2025-04-30     28
2025-05-31     35
2025-06-30     26
2025-07-31     32
2025-08-31     32
2025-09-30     31
2025-10-31     29
2025-11-30     27
2025-12-31     25
2026-01-31    791
2026-02-28    797
2026-03-31    813
2026-04-30    802
2026-05-31    808
Name: count, dtype: int64

HDFC Bank unclassified rows: 35
      statement_date                scheme_name
0         2026-05-31         SBI Large Cap Fund
182       2026-05-31  SBI Large and Midcap Fund
260   

In [12]:
from tqdm import tqdm
tqdm.pandas()

def get_cap_category_v2(isin, stmt_date, cap_lookup):
    # First try: exact window match
    match = cap_lookup[
        (cap_lookup["isin"] == isin) &
        (cap_lookup["window_start"] <= stmt_date) &
        (cap_lookup["window_end"]   >= stmt_date)
    ]
    if len(match) > 0:
        return match.iloc[0]["cap_category"], "exact"

    # Second try: if date is beyond the last window (i.e. 2026),
    # fall back to the most recent available window for that ISIN
    future_match = cap_lookup[cap_lookup["isin"] == isin].sort_values("window_end", ascending=False)
    if len(future_match) > 0:
        return future_match.iloc[0]["cap_category"], "proxied_latest"

    # No match at all — genuinely unlisted instrument
    return "Unclassified", "not_found"

print("Applying cap classification v2 (takes ~3 mins)...")
results = holdings.progress_apply(
    lambda row: get_cap_category_v2(row["isin"], row["statement_date"], cap_lookup),
    axis=1
)

holdings["cap_category"]    = results.apply(lambda x: x[0])
holdings["cap_match_type"]  = results.apply(lambda x: x[1])

print("\nCap category distribution:")
print(holdings["cap_category"].value_counts())

print("\nMatch type breakdown:")
print(holdings["cap_match_type"].value_counts())

print(f"\nTrue unclassified (not_found): {holdings['cap_match_type'].eq('not_found').sum()} rows")
print(f"Proxied 2026 rows: {holdings['cap_match_type'].eq('proxied_latest').sum()} rows")

Applying cap classification v2 (takes ~3 mins)...


100%|██████████| 22461/22461 [03:13<00:00, 116.21it/s]


Cap category distribution:
cap_category
Small Cap       8126
Mid Cap         7030
Large Cap       6632
Unclassified     673
Name: count, dtype: int64

Match type breakdown:
cap_match_type
exact             17900
proxied_latest     3888
not_found           673
Name: count, dtype: int64

True unclassified (not_found): 673 rows
Proxied 2026 rows: 3888 rows


In [13]:
out_path = "/content/drive/MyDrive/StyleDrift_Project/processed/holdings_with_cap.csv"
holdings.to_csv(out_path, index=False)

print(f"✓ Saved → {out_path}")
print(f"Shape: {holdings.shape}")
print(f"Columns: {holdings.columns.tolist()}")

✓ Saved → /content/drive/MyDrive/StyleDrift_Project/processed/holdings_with_cap.csv
Shape: (22461, 12)
Columns: ['instrument_name', 'isin', 'sector', 'quantity', 'market_value_lakhs', 'pct_to_aum', 'scheme_name', 'statement_date', 'amc', 'file_label', 'cap_category', 'cap_match_type']
